# Berth occupancy and waiting time

Why do small increases in berth occupancy sometimes cause very large increases in waiting time? Plot the M/M/n **occupancy hockey stick** for 1, 2 and 4 shared berths. Dense analytical Erlang C curves show the steady-state trend, while independent OpenQTSim simulations check selected points.

Arrivals are Poisson, service times are exponential, and all berths have the same service rate. There is one FIFO queue, unlimited waiting space, no quay-length limit and no reneging.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import openqtsim
from IPython.display import display
from openqtsim.mt_engine import create_tasks, run_tasks, statistics_worker, mmc_waiting_time_factor

print(f'This notebook uses OpenQTSim {openqtsim.__version__}')

This notebook uses OpenQTSim v0.5.1


## 1. Define occupancy levels and simulation effort

Keep the service rate **per berth** fixed. At occupancy $\rho$, the total arrival rate is $\lambda = \rho n \mu_{server}$. Thus, comparisons at the **same occupancy** allow more traffic when more berths are present; they are not comparisons at the same traffic demand.

`nr_arr` customers are simulated in each replication. Each plotted combination of berth count and occupancy therefore uses `nr_arr * nr_replications` customers. Runs start empty; increasing replications alone does not eliminate finite-run start-up bias. The master seed yields separate arrival and service seeds for every task.

The defaults use 8 replications of 100,000 customers per point (800,000 customers per combination, 9.6 million overall). This larger budget helps at 90% occupancy; expect several minutes of serial computation. For a quick preview, reduce these values, but do not interpret a short high-occupancy run as a precise steady-state estimate.

In [2]:
servers = [1, 2, 4]
mu_server = 1.0  # vessels/hour per berth
utilisations = [0.3, 0.5, 0.7, 0.9]  # simulated points
rho_curve = np.linspace(0.05, 0.95, 181)
nr_arr = 100_000
nr_replications = 8
seed = 2024
client = None  # optionally supply an existing Dask Client

if mu_server <= 0 or nr_replications < 2 or not all(0 < rho < 1 for rho in utilisations):
    raise ValueError('Use positive service capacity, stable occupancies and at least two replications.')
print(f'{nr_arr * nr_replications:,} customers per simulated combination')
print(f'{len(servers) * len(utilisations) * nr_replications} independent simulations')

800,000 customers per simulated combination
96 independent simulations


## 2. Simulate the selected points

`create_tasks` interprets `mu` as **total system capacity**, so pass `n * mu_server` for each berth count. `statistics_worker` returns mean waiting times in hours. Tasks run serially by default; passing a Dask client changes scheduling, not the seeded results.

In [3]:
scenario_seeds = [int(child.generate_state(1)[0]) for child in np.random.SeedSequence(seed).spawn(len(servers))]
tasks = []
for n, scenario_seed in zip(servers, scenario_seeds):
    tasks.extend(create_tasks('M', 'M', [n], utilisations, nr_arr, mu=n * mu_server,
                              nr_replications=nr_replications, seed=scenario_seed))
results = pd.DataFrame(run_tasks(tasks, client, func=statistics_worker))
results['utilisation'] = (results['lam'] / results['mu']).round(10)
points = results.groupby(['c', 'utilisation'])['W_q'].agg(['mean', 'std', 'count'])
points['standard error'] = points['std'] / np.sqrt(points['count'])
points['analytical'] = [mmc_waiting_time_factor(int(n), rho) / mu_server for n, rho in points.index]
display(points[['mean', 'standard error', 'analytical']].rename(columns={'mean': 'simulated W_q (hours)'}).round(4))

simulated W_q (hours)  standard error  analytical
c utilisation                                                   
1 0.3                         0.4258          0.0026      0.4286
  0.5                         0.9934          0.0066      1.0000
  0.7                         2.3383          0.0208      2.3333
  0.9                         8.6503          0.1793      9.0000
2 0.3                         0.0992          0.0009      0.0989
  0.5                         0.3285          0.0033      0.3333
  0.7                         0.9711          0.0069      0.9608
  0.9                         4.2839          0.1118      4.2632
4 0.3                         0.0129          0.0003      0.0132
  0.5                         0.0853          0.0009      0.0870
  0.7                         0.3598          0.0042      0.3572
  0.9                         1.9760          0.0493      1.9694

## 3. The hockey-stick curve

Lines are exact steady-state results; markers are simulation averages with **one standard error** across replications. Error bars describe uncertainty in the estimated mean, not the distribution of individual waits, and do not include start-up bias. Near full occupancy, waits are strongly correlated and longer runs are especially important.

All curves grow without bound as occupancy approaches 100%; there is no finite steady-state mean at or above full utilisation. A logarithmic scale is deliberately avoided here so that the steep rise is visible directly.

In [4]:
fig = go.Figure()
colors = ['#0072B2', '#D55E00', '#009E73']
for i, n in enumerate(servers):
    color = colors[i % len(colors)]
    curve = [mmc_waiting_time_factor(n, rho) / mu_server for rho in rho_curve]
    measured = points.xs(n, level='c')
    fig.add_scatter(x=100 * rho_curve, y=curve, mode='lines', line=dict(color=color),
                    name=f'{n} berth(s): Erlang C', legendgroup=str(n))
    fig.add_scatter(x=100 * measured.index.to_numpy(), y=measured['mean'], mode='markers',
                    marker=dict(color=color, size=9), name=f'{n} berth(s): simulation +/- 1 SE',
                    error_y=dict(type='data', array=measured['standard error']), legendgroup=str(n))
fig.update_layout(title='Waiting rises sharply as berth occupancy approaches 100%',
                  xaxis_title='Berth occupancy / server utilisation (%)',
                  yaxis_title='Mean waiting time W_q (hours)', template='plotly_white', height=600)
fig.show()

## 4. Adding a berth at fixed traffic is a different comparison

In the plot, each curve changes the arrival rate to reach the chosen occupancy. For investment planning, instead keep $\lambda$ fixed: an additional berth **reduces occupancy as well as improving pooling**. The following analytical table demonstrates this with 0.8 vessels/hour at every berth count. Increase `fixed_arrival_rate` to see which layouts become unstable.

In [5]:
fixed_arrival_rate = 0.8  # total vessels/hour in every layout
if fixed_arrival_rate <= 0:
    raise ValueError('fixed_arrival_rate must be positive.')
fixed_traffic = pd.DataFrame({'berths': servers}).set_index('berths')
fixed_traffic['arrival rate (vessels/hour)'] = fixed_arrival_rate
fixed_traffic['occupancy'] = fixed_arrival_rate / (fixed_traffic.index * mu_server)
fixed_traffic['stable'] = fixed_traffic['occupancy'] < 1
fixed_traffic['analytical W_q (hours)'] = [
    mmc_waiting_time_factor(int(n), row['occupancy']) / mu_server for n, row in fixed_traffic.iterrows()
]
display(fixed_traffic.round(4))

,arrival rate (vessels/hour),occupancy,stable,analytical W_q (hours)
berths,,,,
1,0.8,0.8,True,4.0000
2,0.8,0.4,True,0.1905
4,0.8,0.2,True,0.0030


## 5. Questions to explore

* With one berth, raising occupancy from 50% to 90% raises the analytical wait from 1 to 9 mean service times. What is the corresponding change with four berths?
* How much traffic can each layout handle while keeping the mean wait below one hour?
* Increase `nr_arr` and `nr_replications`: do the high-occupancy estimates stabilise?

There is no universal 'safe' occupancy threshold: it depends on acceptable delays, variability, pooling and costs. **2003 - Berth planning by cost** balances waiting against the cost of additional berths.